
# Buy Fee Model V12A.1 — Corrected Buyer Event Study + Matched-Control Buyer DiD

## Objective

V12A successfully linked buyer data to fee events, but the original midpoint-based logic produced **0 events with full 4 PRE / 4 POST biweekly periods**.

V12A.1 fixes that problem.

### Corrected event windows

For each fee-change date:

- **PRE:** calendar days `Change Date - 56` through `Change Date - 1`
- **POST:** calendar days `Change Date + 1` through `Change Date + 56`

A biweekly buyer period is included when its date interval **overlaps** the relevant PRE or POST window.

### Coverage requirement

A treated or control location must have:

- at least **3 valid biweekly periods PRE**
- at least **3 valid biweekly periods POST**

### Matched-control Buyer DiD

For every treated fee-change segment, V12A.1 uses the matched controls created in V8.2.

For each buyer metric:

`Buyer DiD = Treated PRE→POST Change - Average Control PRE→POST Change`

Metrics:

- Unique Buyers
- Active Buyers
- Fallout Buyers
- Dark Buyers
- Reactivated Buyers
- Net New Buyers
- Purchase recency:
  - Average Days Since Last Purchase
  - Average Days Since Previous Location Purchase

For buyer counts, the notebook calculates:
- absolute change
- percent change
- matched-control DiD of both

For purchase recency, the primary metric is:
- change in days
- matched-control DiD in days

## Important

Buyer data is location-level. The Buy Fee event remains Location × Lane × Price Tier.  
Therefore buyer behavior is shared across segment events occurring at the same location/date, but each segment retains its own matched controls and fee-response outcome.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import gc
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

DATA_FOLDER = Path(
    r"C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion"
)

V82_DIR = DATA_FOLDER / "buy_fee_model_v8_2_outputs"
V12A_DIR = DATA_FOLDER / "buy_fee_model_v12a_outputs"

# V8.2 matched controls
MATCHED_PAIRS_FILE = V82_DIR / "03_MATCHED_CONTROL_PAIRS.csv"

# V8.2 event-level DiD outcomes
DID_FILE = V82_DIR / "04_DID_EVENT_LEVEL_RESULTS.csv"

# Small aggregate already created by V12A
BUYER_AGG_FILE = V12A_DIR / "01_BUYER_BIWEEKLY_AGGREGATE.csv"

OUTPUT_DIR = DATA_FOLDER / "buy_fee_model_v12a_1_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = [10, 15, 20]
SCENARIO_TOLERANCE = 2.5

# Corrected calendar-distance windows
PRE_DAYS = 56
POST_DAYS = 56

# Minimum usable buyer coverage
MIN_PRE_PERIODS = 3
MIN_POST_PERIODS = 3

# Balanced success thresholds inherited from V10/V12A
RPU_SUCCESS_MIN = 0.0
CONVERSION_SAFE_MIN_PP = -2.0
VOLUME_SAFE_MIN_PCT = -0.10

print("=" * 80)
print("V12A.1 INPUT CHECK")
print("=" * 80)

for p in [
    MATCHED_PAIRS_FILE,
    DID_FILE,
    BUYER_AGG_FILE
]:
    print(f"{p.name:<40} {p.exists()}")

missing = [
    p for p in [
        MATCHED_PAIRS_FILE,
        DID_FILE,
        BUYER_AGG_FILE
    ]
    if not p.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required file(s):\n"
        + "\n".join(str(p) for p in missing)
    )

print("\nOutput folder:")
print(OUTPUT_DIR)


V12A.1 INPUT CHECK
03_MATCHED_CONTROL_PAIRS.csv             True
04_DID_EVENT_LEVEL_RESULTS.csv           True
01_BUYER_BIWEEKLY_AGGREGATE.csv          True

Output folder:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v12a_1_outputs


## 1. Load the small buyer biweekly aggregate

In [2]:

buyer = pd.read_csv(
    BUYER_AGG_FILE,
    low_memory=True
)

required_buyer = [
    "VRLOCATION",
    "BiWeekly_Start",
    "BiWeekly_End",
    "Unique_Buyers",
    "Active_Buyers",
    "Fallout_Buyers",
    "Dark_Buyers",
    "Reactivated_Buyers",
    "Net_New_Buyers",
    "Avg_Days_Since_Last_Purchase",
    "Avg_Days_Since_Previous_Location_Purchase",
]

missing = [
    c for c in required_buyer
    if c not in buyer.columns
]

if missing:
    raise ValueError(
        "Buyer aggregate is missing required columns:\n"
        + "\n".join(missing)
    )

buyer["VRLOCATION"] = (
    buyer["VRLOCATION"]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.upper()
)

buyer["BiWeekly_Start"] = pd.to_datetime(
    buyer["BiWeekly_Start"],
    errors="coerce"
)

buyer["BiWeekly_End"] = pd.to_datetime(
    buyer["BiWeekly_End"],
    errors="coerce"
)

for c in required_buyer[3:]:
    buyer[c] = pd.to_numeric(
        buyer[c],
        errors="coerce"
    )

buyer = buyer[
    buyer["VRLOCATION"].ne("")
    & buyer["BiWeekly_Start"].notna()
    & buyer["BiWeekly_End"].notna()
]

print("Buyer aggregate rows:", f"{len(buyer):,}")
print("Locations:", buyer["VRLOCATION"].nunique())
print(
    "Date range:",
    buyer["BiWeekly_Start"].min(),
    "to",
    buyer["BiWeekly_End"].max()
)

display(buyer.head())


Buyer aggregate rows: 3,498
Locations: 49
Date range: 2019-12-30 00:00:00 to 2022-09-24 00:00:00


,VRLOCATION,BiWeekly_Start,BiWeekly_End,Unique_Buyers,Active_Buyers,Fallout_Buyers,Dark_Buyers,Reactivated_Buyers,Net_New_Buyers,Days_Since_Last_Sum,Days_Since_Last_Count,Days_Since_Prev_Loc_Sum,Days_Since_Prev_Loc_Count,Avg_Days_Since_Last_Purchase,Avg_Days_Since_Previous_Location_Purchase,Active_Rate,Fallout_Rate,Dark_Rate,Reactivated_Rate,Net_New_Rate
0,ALBANY,2019-12-30,2020-01-12,149,45,26,74,4,0,18634.0,149,4903.0,96,125.060403,51.072917,0.302013,0.174497,0.496644,0.026846,0.000000
1,ALBANY,2020-01-13,2020-01-26,149,50,22,72,5,0,18898.0,149,4707.0,98,126.832215,48.030612,0.335570,0.147651,0.483221,0.033557,0.000000
2,ALBANY,2020-01-27,2020-02-09,152,47,27,73,5,2,19997.0,152,4737.0,98,131.559211,48.336735,0.309211,0.177632,0.480263,0.032895,0.013158
3,ALBANY,2020-02-10,2020-02-23,154,46,29,76,3,1,21130.0,154,4469.0,98,137.207792,45.602041,0.298701,0.188312,0.493506,0.019481,0.006494
4,ALBANY,2020-02-24,2020-03-08,156,51,28,73,4,1,21703.0,156,4490.0,99,139.121795,45.353535,0.326923,0.179487,0.467949,0.025641,0.006410


## 2. Load V8.2 matched controls and event outcomes

In [3]:

pairs = pd.read_csv(
    MATCHED_PAIRS_FILE,
    low_memory=True
)

did = pd.read_csv(
    DID_FILE,
    low_memory=True
)

required_pairs = [
    "Segment_Change_ID",
    "Corporate_Change_ID",
    "BUYFEE_CHANGE_AMOUNT",
    "Treated_Location",
    "Control_Rank",
    "Control_Location",
]

missing_pairs = [
    c for c in required_pairs
    if c not in pairs.columns
]

if missing_pairs:
    raise ValueError(
        "Matched-pairs file is missing:\n"
        + "\n".join(missing_pairs)
    )

# Normalize location keys
for c in [
    "Treated_Location",
    "Control_Location"
]:
    pairs[c] = (
        pairs[c]
        .astype("string")
        .fillna("")
        .str.strip()
        .str.upper()
    )

pairs["Change_Date"] = pd.to_datetime(
    pairs["Corporate_Change_ID"],
    errors="coerce"
)

pairs["BUYFEE_CHANGE_AMOUNT"] = pd.to_numeric(
    pairs["BUYFEE_CHANGE_AMOUNT"],
    errors="coerce"
)

# Keep only +10 / +15 / +20 scenarios
scenario_mask = pd.Series(
    False,
    index=pairs.index
)

for fee in SCENARIOS:
    scenario_mask |= (
        pairs["BUYFEE_CHANGE_AMOUNT"]
        .between(
            fee - SCENARIO_TOLERANCE,
            fee + SCENARIO_TOLERANCE
        )
    )

pairs = pairs[
    scenario_mask
    & pairs["Change_Date"].notna()
    & pairs["Treated_Location"].ne("")
    & pairs["Control_Location"].ne("")
].copy()

# Event-level outcome fields
did_keep = [
    c for c in [
        "Segment_Change_ID",
        "DiD_RPU_Change_Dollar",
        "DiD_Conversion_Change_pp",
        "DiD_Volume_Change_Pct",
        "Treated_Lane",
        "Treated_Price_Tier",
    ]
    if c in did.columns
]

did_small = (
    did[did_keep]
    .drop_duplicates(
        subset=["Segment_Change_ID"]
    )
)

pairs = pairs.merge(
    did_small,
    on="Segment_Change_ID",
    how="left"
)

for c in [
    "DiD_RPU_Change_Dollar",
    "DiD_Conversion_Change_pp",
    "DiD_Volume_Change_Pct"
]:
    if c in pairs.columns:
        pairs[c] = pd.to_numeric(
            pairs[c],
            errors="coerce"
        )

print("Matched pair rows retained:", f"{len(pairs):,}")
print(
    "Treated events:",
    pairs["Segment_Change_ID"].nunique()
)
print(
    "Corporate change dates:",
    pairs["Corporate_Change_ID"].nunique()
)


Matched pair rows retained: 2,425
Treated events: 810
Corporate change dates: 29


## 3. Calendar-distance PRE/POST helper

In [4]:

COUNT_METRICS = [
    "Unique_Buyers",
    "Active_Buyers",
    "Fallout_Buyers",
    "Dark_Buyers",
    "Reactivated_Buyers",
    "Net_New_Buyers",
]

RECENCY_METRICS = [
    "Avg_Days_Since_Last_Purchase",
    "Avg_Days_Since_Previous_Location_Purchase",
]

ALL_METRICS = (
    COUNT_METRICS
    + RECENCY_METRICS
)

# Build a location lookup once
buyer_by_location = {
    loc: grp.sort_values(
        "BiWeekly_Start"
    ).reset_index(drop=True)
    for loc, grp in buyer.groupby(
        "VRLOCATION",
        sort=False
    )
}

def overlap_periods(
    location,
    window_start,
    window_end
):
    """
    Return buyer biweekly periods whose date interval overlaps
    the requested calendar window.

    Overlap rule:
      Period_Start <= Window_End
      AND
      Period_End >= Window_Start
    """

    d = buyer_by_location.get(
        location
    )

    if d is None or d.empty:
        return pd.DataFrame()

    return d[
        (d["BiWeekly_Start"] <= window_end)
        &
        (d["BiWeekly_End"] >= window_start)
    ]

def summarize_location_event(
    location,
    change_date
):
    """
    Summarize location-level buyer behavior for:
    PRE  = change_date - 56 through change_date - 1
    POST = change_date + 1 through change_date + 56
    """

    pre_start = (
        change_date
        - pd.Timedelta(days=PRE_DAYS)
    )

    pre_end = (
        change_date
        - pd.Timedelta(days=1)
    )

    post_start = (
        change_date
        + pd.Timedelta(days=1)
    )

    post_end = (
        change_date
        + pd.Timedelta(days=POST_DAYS)
    )

    pre = overlap_periods(
        location,
        pre_start,
        pre_end
    )

    post = overlap_periods(
        location,
        post_start,
        post_end
    )

    result = {
        "Pre_Periods": pre["BiWeekly_Start"].nunique()
        if not pre.empty
        else 0,

        "Post_Periods": post["BiWeekly_Start"].nunique()
        if not post.empty
        else 0,
    }

    for metric in ALL_METRICS:

        pre_val = (
            pre[metric].mean()
            if not pre.empty
            else np.nan
        )

        post_val = (
            post[metric].mean()
            if not post.empty
            else np.nan
        )

        result[f"Pre_{metric}"] = pre_val
        result[f"Post_{metric}"] = post_val

        result[f"Abs_Change_{metric}"] = (
            post_val - pre_val
            if pd.notna(pre_val)
            and pd.notna(post_val)
            else np.nan
        )

        if metric in COUNT_METRICS:

            result[f"Pct_Change_{metric}"] = (
                (post_val - pre_val) / pre_val
                if pd.notna(pre_val)
                and pre_val != 0
                and pd.notna(post_val)
                else np.nan
            )

    result["Coverage_Eligible"] = (
        result["Pre_Periods"] >= MIN_PRE_PERIODS
        and result["Post_Periods"] >= MIN_POST_PERIODS
    )

    return result


## 4. Build treated/control buyer response for each matched pair

In [5]:

pair_rows = []

# Cache repeated location x change-date summaries.
summary_cache = {}

def cached_summary(
    location,
    change_date
):
    key = (
        location,
        pd.Timestamp(change_date).date()
    )

    if key not in summary_cache:
        summary_cache[key] = (
            summarize_location_event(
                location,
                change_date
            )
        )

    return summary_cache[key]

for row_num, (_, r) in enumerate(
    pairs.iterrows(),
    start=1
):

    change_date = pd.Timestamp(
        r["Change_Date"]
    )

    t = cached_summary(
        r["Treated_Location"],
        change_date
    )

    c = cached_summary(
        r["Control_Location"],
        change_date
    )

    out = {
        "Segment_Change_ID":
            r["Segment_Change_ID"],

        "Corporate_Change_ID":
            r["Corporate_Change_ID"],

        "Change_Date":
            change_date,

        "BUYFEE_CHANGE_AMOUNT":
            r["BUYFEE_CHANGE_AMOUNT"],

        "Treated_Location":
            r["Treated_Location"],

        "Control_Rank":
            r["Control_Rank"],

        "Control_Location":
            r["Control_Location"],

        "Treated_Lane":
            r.get(
                "Treated_Lane",
                np.nan
            ),

        "Treated_Price_Tier":
            r.get(
                "Treated_Price_Tier",
                np.nan
            ),

        "DiD_RPU_Change_Dollar":
            r.get(
                "DiD_RPU_Change_Dollar",
                np.nan
            ),

        "DiD_Conversion_Change_pp":
            r.get(
                "DiD_Conversion_Change_pp",
                np.nan
            ),

        "DiD_Volume_Change_Pct":
            r.get(
                "DiD_Volume_Change_Pct",
                np.nan
            ),

        "T_Pre_Periods":
            t["Pre_Periods"],

        "T_Post_Periods":
            t["Post_Periods"],

        "C_Pre_Periods":
            c["Pre_Periods"],

        "C_Post_Periods":
            c["Post_Periods"],

        "T_Coverage_Eligible":
            t["Coverage_Eligible"],

        "C_Coverage_Eligible":
            c["Coverage_Eligible"],
    }

    for metric in ALL_METRICS:

        out[f"T_Pre_{metric}"] = (
            t[f"Pre_{metric}"]
        )

        out[f"T_Post_{metric}"] = (
            t[f"Post_{metric}"]
        )

        out[f"T_Abs_Change_{metric}"] = (
            t[f"Abs_Change_{metric}"]
        )

        out[f"C_Pre_{metric}"] = (
            c[f"Pre_{metric}"]
        )

        out[f"C_Post_{metric}"] = (
            c[f"Post_{metric}"]
        )

        out[f"C_Abs_Change_{metric}"] = (
            c[f"Abs_Change_{metric}"]
        )

        # Absolute DiD
        out[f"Buyer_DiD_Abs_{metric}"] = (
            t[f"Abs_Change_{metric}"]
            - c[f"Abs_Change_{metric}"]
            if pd.notna(
                t[f"Abs_Change_{metric}"]
            )
            and pd.notna(
                c[f"Abs_Change_{metric}"]
            )
            else np.nan
        )

        if metric in COUNT_METRICS:

            out[f"T_Pct_Change_{metric}"] = (
                t[f"Pct_Change_{metric}"]
            )

            out[f"C_Pct_Change_{metric}"] = (
                c[f"Pct_Change_{metric}"]
            )

            # Percent-change DiD
            out[f"Buyer_DiD_Pct_{metric}"] = (
                t[f"Pct_Change_{metric}"]
                - c[f"Pct_Change_{metric}"]
                if pd.notna(
                    t[f"Pct_Change_{metric}"]
                )
                and pd.notna(
                    c[f"Pct_Change_{metric}"]
                )
                else np.nan
            )

    pair_rows.append(out)

    if row_num % 1000 == 0:
        print(
            f"Processed {row_num:,} matched pairs"
        )

buyer_pair = pd.DataFrame(
    pair_rows
)

buyer_pair["Pair_Coverage_Eligible"] = (
    buyer_pair["T_Coverage_Eligible"]
    &
    buyer_pair["C_Coverage_Eligible"]
)

buyer_pair.to_csv(
    OUTPUT_DIR
    / "01_BUYER_MATCHED_PAIR_DID.csv",
    index=False
)

print("\nMatched buyer-pair rows:", f"{len(buyer_pair):,}")

print(
    "Coverage-eligible pairs:",
    f"{buyer_pair['Pair_Coverage_Eligible'].sum():,}"
)

print(
    "Coverage rate:",
    round(
        buyer_pair[
            "Pair_Coverage_Eligible"
        ].mean(),
        4
    )
)


Processed 1,000 matched pairs
Processed 2,000 matched pairs

Matched buyer-pair rows: 2,425
Coverage-eligible pairs: 0
Coverage rate: 0.0


## 5. Aggregate matched controls to one Buyer DiD row per treated event

In [8]:
# ============================================================
# FIX V12A.1 DATE ALIGNMENT
# USE TRUE BUYFEE_CHANGE_DATE, NOT Corporate_Change_ID
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import gc


print("=" * 80)
print("FIXING BUYER / FEE EVENT DATE ALIGNMENT")
print("=" * 80)


# ============================================================
# 1. FILE PATHS
# ============================================================

V61_DIR = (
    DATA_FOLDER
    / "buy_fee_model_dataset_v6_1"
)

STRICT_EVENT_FILE = (
    V61_DIR
    / "09_MODEL_READY_STRICT_V6_1.csv"
)

MATCHED_PAIRS_FILE = (
    V82_DIR
    / "03_MATCHED_CONTROL_PAIRS.csv"
)


print("Strict fee event file:")
print(STRICT_EVENT_FILE)

print(
    "Exists:",
    STRICT_EVENT_FILE.exists()
)


print("\nMatched pair file:")
print(MATCHED_PAIRS_FILE)

print(
    "Exists:",
    MATCHED_PAIRS_FILE.exists()
)


if not STRICT_EVENT_FILE.exists():

    raise FileNotFoundError(
        f"Missing V6.1 strict event file:\n"
        f"{STRICT_EVENT_FILE}"
    )


if not MATCHED_PAIRS_FILE.exists():

    raise FileNotFoundError(
        f"Missing V8.2 matched pair file:\n"
        f"{MATCHED_PAIRS_FILE}"
    )


# ============================================================
# 2. LOAD MATCHED PAIRS
# ============================================================

pairs = pd.read_csv(
    MATCHED_PAIRS_FILE,
    low_memory=True
)


print(
    "\nMatched pair rows:",
    f"{len(pairs):,}"
)


# ============================================================
# 3. LOAD ONLY REQUIRED FEE EVENT COLUMNS
# ============================================================

strict_header = pd.read_csv(
    STRICT_EVENT_FILE,
    nrows=0
)


print(
    "\nV6.1 strict event columns containing DATE:"
)

for c in strict_header.columns:

    if "DATE" in str(c).upper():

        print(
            " -",
            c
        )


required_strict_cols = [
    "Segment_Change_ID",
    "BUYFEE_CHANGE_DATE",
]


missing = [
    c
    for c in required_strict_cols
    if c not in strict_header.columns
]


if missing:

    raise ValueError(
        "V6.1 strict event file is missing:\n"
        + "\n".join(missing)
    )


strict_dates = pd.read_csv(
    STRICT_EVENT_FILE,
    usecols=required_strict_cols,
    low_memory=True
)


# ============================================================
# 4. PARSE TRUE BUY FEE CHANGE DATE
# ============================================================

strict_dates[
    "BUYFEE_CHANGE_DATE"
] = pd.to_datetime(
    strict_dates[
        "BUYFEE_CHANGE_DATE"
    ],
    errors="coerce"
)


strict_dates = (
    strict_dates
    .dropna(
        subset=[
            "Segment_Change_ID",
            "BUYFEE_CHANGE_DATE"
        ]
    )
    .drop_duplicates(
        subset=[
            "Segment_Change_ID"
        ],
        keep="first"
    )
)


print(
    "\nStrict events with valid change date:",
    f"{len(strict_dates):,}"
)


print(
    "Fee event date range:",
    strict_dates[
        "BUYFEE_CHANGE_DATE"
    ].min(),
    "to",
    strict_dates[
        "BUYFEE_CHANGE_DATE"
    ].max()
)


# ============================================================
# 5. REMOVE ANY OLD / INCORRECT CHANGE DATE
# ============================================================

for c in [
    "Change_Date",
    "BUYFEE_CHANGE_DATE"
]:

    if c in pairs.columns:

        pairs.drop(
            columns=[c],
            inplace=True
        )


# ============================================================
# 6. JOIN TRUE DATE BY SEGMENT_CHANGE_ID
# ============================================================

pairs = pairs.merge(
    strict_dates,
    on="Segment_Change_ID",
    how="left",
    validate="many_to_one"
)


pairs.rename(
    columns={
        "BUYFEE_CHANGE_DATE":
            "Change_Date"
    },
    inplace=True
)


pairs[
    "Change_Date"
] = pd.to_datetime(
    pairs[
        "Change_Date"
    ],
    errors="coerce"
)


# ============================================================
# 7. NORMALIZE LOCATIONS
# ============================================================

for c in [
    "Treated_Location",
    "Control_Location"
]:

    pairs[c] = (
        pairs[c]
        .astype("string")
        .fillna("")
        .str.strip()
        .str.upper()
    )


# ============================================================
# 8. MAKE BUY FEE CHANGE AMOUNT NUMERIC
# ============================================================

pairs[
    "BUYFEE_CHANGE_AMOUNT"
] = pd.to_numeric(
    pairs[
        "BUYFEE_CHANGE_AMOUNT"
    ],
    errors="coerce"
)


# ============================================================
# 9. KEEP ONLY +10 / +15 / +20
# ============================================================

scenario_mask = pd.Series(
    False,
    index=pairs.index
)


for fee in SCENARIOS:

    scenario_mask |= (
        pairs[
            "BUYFEE_CHANGE_AMOUNT"
        ]
        .between(
            fee - SCENARIO_TOLERANCE,
            fee + SCENARIO_TOLERANCE
        )
    )


pairs = pairs[
    scenario_mask
    &
    pairs["Change_Date"].notna()
    &
    pairs["Treated_Location"].ne("")
    &
    pairs["Control_Location"].ne("")
]


gc.collect()


# ============================================================
# 10. COMPARE BUYER DATE RANGE VS FEE DATE RANGE
# ============================================================

buyer_min = (
    buyer[
        "BiWeekly_Start"
    ].min()
)

buyer_max = (
    buyer[
        "BiWeekly_End"
    ].max()
)

fee_min = (
    pairs[
        "Change_Date"
    ].min()
)

fee_max = (
    pairs[
        "Change_Date"
    ].max()
)


print(
    "\n"
    + "=" * 80
)

print(
    "DATE RANGE DIAGNOSTIC"
)

print(
    "=" * 80
)


print(
    "Buyer data:",
    buyer_min,
    "to",
    buyer_max
)


print(
    "Fee events:",
    fee_min,
    "to",
    fee_max
)


# ============================================================
# 11. CHECK WHETHER THE RANGES OVERLAP
# ============================================================

date_ranges_overlap = (
    buyer_min <= fee_max
    and buyer_max >= fee_min
)


print(
    "\nDate ranges overlap:",
    date_ranges_overlap
)


# ============================================================
# 12. CHECK EVENT COVERAGE INSIDE BUYER HISTORY
# ============================================================

pairs[
    "Fee_Date_Inside_Buyer_History"
] = (
    pairs[
        "Change_Date"
    ].between(
        buyer_min,
        buyer_max
    )
)


print(
    "\nMatched pair rows:",
    f"{len(pairs):,}"
)


print(
    "Pairs whose fee date is inside buyer history:",
    f"{pairs['Fee_Date_Inside_Buyer_History'].sum():,}"
)


print(
    "Percent inside buyer history:",
    round(
        pairs[
            "Fee_Date_Inside_Buyer_History"
        ].mean()
        * 100,
        2
    ),
    "%"
)


print(
    "\nUnique treated events:",
    f"{pairs['Segment_Change_ID'].nunique():,}"
)


inside_events = (
    pairs.loc[
        pairs[
            "Fee_Date_Inside_Buyer_History"
        ],
        "Segment_Change_ID"
    ]
    .nunique()
)


print(
    "Treated events inside buyer history:",
    f"{inside_events:,}"
)


# ============================================================
# 13. LOCATION MATCH DIAGNOSTIC
# ============================================================

buyer_locations = set(
    buyer[
        "VRLOCATION"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .unique()
)


treated_locations = set(
    pairs[
        "Treated_Location"
    ]
    .dropna()
    .unique()
)


control_locations = set(
    pairs[
        "Control_Location"
    ]
    .dropna()
    .unique()
)


treated_matched = (
    treated_locations
    & buyer_locations
)

control_matched = (
    control_locations
    & buyer_locations
)


print(
    "\n"
    + "=" * 80
)

print(
    "LOCATION MATCH DIAGNOSTIC"
)

print(
    "=" * 80
)


print(
    "Buyer locations:",
    len(
        buyer_locations
    )
)


print(
    "Treated locations:",
    len(
        treated_locations
    )
)


print(
    "Treated locations found in buyer data:",
    len(
        treated_matched
    )
)


print(
    "Control locations:",
    len(
        control_locations
    )
)


print(
    "Control locations found in buyer data:",
    len(
        control_matched
    )
)


# ============================================================
# 14. SHOW MISSING LOCATIONS
# ============================================================

missing_treated_locations = sorted(
    treated_locations
    - buyer_locations
)


missing_control_locations = sorted(
    control_locations
    - buyer_locations
)


if missing_treated_locations:

    print(
        "\nTreated locations missing from buyer data:"
    )

    print(
        missing_treated_locations
    )


if missing_control_locations:

    print(
        "\nControl locations missing from buyer data:"
    )

    print(
        missing_control_locations
    )


# ============================================================
# 15. FINAL VALIDATION
# ============================================================

print(
    "\n"
    + "=" * 80
)

print(
    "TRUE BUY FEE DATE ALIGNMENT COMPLETE"
)

print(
    "=" * 80
)


if not date_ranges_overlap:

    print(
        "\nIMPORTANT:"
    )

    print(
        "The buyer-history dates and Buy Fee event dates "
        "do not overlap."
    )

    print(
        "Buyer DiD cannot be calculated until overlapping "
        "buyer history is available."
    )


elif (
    pairs[
        "Fee_Date_Inside_Buyer_History"
    ].sum()
    == 0
):

    print(
        "\nIMPORTANT:"
    )

    print(
        "The broad ranges overlap, but none of the selected "
        "+$10 / +$15 / +$20 fee events fall inside the buyer-history window."
    )


else:

    print(
        "\nPASS:"
    )

    print(
        "Fee events and buyer history overlap."
    )

    print(
        "Now rerun the buyer PRE/POST matched-pair creation cell."
    )

FIXING BUYER / FEE EVENT DATE ALIGNMENT
Strict fee event file:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_dataset_v6_1\09_MODEL_READY_STRICT_V6_1.csv
Exists: True

Matched pair file:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v8_2_outputs\03_MATCHED_CONTROL_PAIRS.csv
Exists: True

Matched pair rows: 5,435

V6.1 strict event columns containing DATE:
 - BUYFEE_CHANGE_DATE
 - Prev_Change_Date
 - Next_Change_Date

Strict events with valid change date: 1,843
Fee event date range: 2023-02-12 00:00:00 to 2026-02-08 00:00:00

DATE RANGE DIAGNOSTIC
Buyer data: 2019-12-30 00:00:00 to 2022-09-24 00:00:00
Fee events: 2023-02-12 00:00:00 to 2026-02-08 00:00:00

Date ranges overlap: False

Matched pair rows: 2,425
Pairs whos

## 6. Buyer DiD by +$10 / +$15 / +$20

In [9]:

scenario_rows = []

for fee in SCENARIOS:

    d = buyer_event_did[
        buyer_event_did[
            "BUYFEE_CHANGE_AMOUNT"
        ].between(
            fee - SCENARIO_TOLERANCE,
            fee + SCENARIO_TOLERANCE
        )
    ].copy()

    row = {
        "Fee_Increase": fee,
        "Events": len(d),
        "Locations":
            d["Treated_Location"].nunique(),
        "Corporate_Change_Dates":
            d["Corporate_Change_ID"].nunique(),
        "Median_Valid_Controls":
            d["Valid_Controls"].median(),
    }

    for metric in COUNT_METRICS:

        row[
            f"Median_Buyer_DiD_Pct_{metric}"
        ] = d[
            f"Buyer_DiD_Pct_{metric}"
        ].median()

        row[
            f"Median_Buyer_DiD_Abs_{metric}"
        ] = d[
            f"Buyer_DiD_Abs_{metric}"
        ].median()

    for metric in RECENCY_METRICS:

        row[
            f"Median_Buyer_DiD_Days_{metric}"
        ] = d[
            f"Buyer_DiD_Abs_{metric}"
        ].median()

    scenario_rows.append(
        row
    )

scenario_summary = pd.DataFrame(
    scenario_rows
)

scenario_summary.to_csv(
    OUTPUT_DIR
    / "03_BUYER_DID_BY_SCENARIO.csv",
    index=False
)

display(scenario_summary)


NameError: name 'buyer_event_did' is not defined

## 7. Successful vs unsuccessful PRE-event buyer profile

In [ ]:

pre_features = [
    f"Pre_{m}"
    for m in ALL_METRICS
]

profile_rows = []

for feature in pre_features:

    success = pd.to_numeric(
        buyer_event_did.loc[
            buyer_event_did[
                "Balanced_Success"
            ],
            feature
        ],
        errors="coerce"
    ).dropna()

    fail = pd.to_numeric(
        buyer_event_did.loc[
            ~buyer_event_did[
                "Balanced_Success"
            ],
            feature
        ],
        errors="coerce"
    ).dropna()

    profile_rows.append({
        "Feature": feature,

        "Successful_N":
            len(success),

        "Failed_N":
            len(fail),

        "Successful_Median":
            success.median()
            if len(success)
            else np.nan,

        "Failed_Median":
            fail.median()
            if len(fail)
            else np.nan,

        "Median_Difference":
            (
                success.median()
                - fail.median()
            )
            if len(success)
            and len(fail)
            else np.nan,
    })

pre_profile = pd.DataFrame(
    profile_rows
)

pre_profile.to_csv(
    OUTPUT_DIR
    / "04_PRE_BUYER_SUCCESS_VS_FAILURE.csv",
    index=False
)

display(pre_profile)


## 8. Does control-adjusted buyer behavior explain fee outcomes?

In [ ]:

mechanism_features = [
    f"Buyer_DiD_Pct_{m}"
    for m in COUNT_METRICS
] + [
    f"Buyer_DiD_Abs_{m}"
    for m in RECENCY_METRICS
]

mechanism_rows = []

for feature in mechanism_features:

    temp = buyer_event_did[
        [
            feature,
            "DiD_RPU_Change_Dollar",
            "DiD_Conversion_Change_pp",
            "DiD_Volume_Change_Pct",
        ]
    ].copy()

    for c in temp.columns:
        temp[c] = pd.to_numeric(
            temp[c],
            errors="coerce"
        )

    temp = temp.dropna(
        subset=[feature]
    )

    if len(temp) < 20:
        continue

    mechanism_rows.append({
        "Buyer_DiD_Feature":
            feature,

        "Events":
            len(temp),

        "Corr_with_RPU":
            temp[feature].corr(
                temp[
                    "DiD_RPU_Change_Dollar"
                ]
            ),

        "Corr_with_Conversion":
            temp[feature].corr(
                temp[
                    "DiD_Conversion_Change_pp"
                ]
            ),

        "Corr_with_Volume":
            temp[feature].corr(
                temp[
                    "DiD_Volume_Change_Pct"
                ]
            ),
    })

mechanism = pd.DataFrame(
    mechanism_rows
)

if not mechanism.empty:

    mechanism[
        "Max_Absolute_Correlation"
    ] = (
        mechanism[
            [
                "Corr_with_RPU",
                "Corr_with_Conversion",
                "Corr_with_Volume"
            ]
        ]
        .abs()
        .max(axis=1)
    )

    mechanism = mechanism.sort_values(
        "Max_Absolute_Correlation",
        ascending=False
    )

mechanism.to_csv(
    OUTPUT_DIR
    / "05_BUYER_DID_MECHANISM_CORRELATIONS.csv",
    index=False
)

display(mechanism)


## 9. Rank PRE-event buyer features for the future predictive model

In [ ]:

ranking_rows = []

for feature in pre_features:

    x = pd.to_numeric(
        buyer_event_did[feature],
        errors="coerce"
    )

    valid = x.notna()

    if valid.sum() < 20:
        continue

    success = (
        buyer_event_did[
            "Balanced_Success"
        ]
        .astype(int)
    )

    def corr_with(col):

        y = pd.to_numeric(
            buyer_event_did[col],
            errors="coerce"
        )

        mask = (
            valid
            & y.notna()
        )

        if mask.sum() < 20:
            return np.nan

        return x[mask].corr(
            y[mask]
        )

    ranking_rows.append({
        "Feature":
            feature,

        "Events":
            int(valid.sum()),

        "Corr_with_Balanced_Success":
            x[valid].corr(
                success[valid]
            ),

        "Corr_with_RPU":
            corr_with(
                "DiD_RPU_Change_Dollar"
            ),

        "Corr_with_Conversion":
            corr_with(
                "DiD_Conversion_Change_pp"
            ),

        "Corr_with_Volume":
            corr_with(
                "DiD_Volume_Change_Pct"
            ),
    })

feature_ranking = pd.DataFrame(
    ranking_rows
)

if not feature_ranking.empty:

    feature_ranking[
        "Max_Absolute_Outcome_Correlation"
    ] = (
        feature_ranking[
            [
                "Corr_with_RPU",
                "Corr_with_Conversion",
                "Corr_with_Volume"
            ]
        ]
        .abs()
        .max(axis=1)
    )

    feature_ranking = (
        feature_ranking
        .sort_values(
            "Max_Absolute_Outcome_Correlation",
            ascending=False
        )
    )

feature_ranking.to_csv(
    OUTPUT_DIR
    / "06_PRE_BUYER_FEATURE_RANKING.csv",
    index=False
)

display(feature_ranking)


## 10. Coverage and readiness summary

In [ ]:

summary = pd.DataFrame([
    {
        "Metric":
            "Matched_Pair_Rows_Input",

        "Value":
            len(pairs)
    },
    {
        "Metric":
            "Coverage_Eligible_Pairs",

        "Value":
            int(
                buyer_pair[
                    "Pair_Coverage_Eligible"
                ].sum()
            )
    },
    {
        "Metric":
            "Coverage_Eligible_Pair_Rate",

        "Value":
            buyer_pair[
                "Pair_Coverage_Eligible"
            ].mean()
    },
    {
        "Metric":
            "Buyer_DiD_Events",

        "Value":
            len(buyer_event_did)
    },
    {
        "Metric":
            "Buyer_DiD_Locations",

        "Value":
            buyer_event_did[
                "Treated_Location"
            ].nunique()
    },
    {
        "Metric":
            "Buyer_DiD_Corporate_Change_Dates",

        "Value":
            buyer_event_did[
                "Corporate_Change_ID"
            ].nunique()
    },
    {
        "Metric":
            "Median_Valid_Controls_Per_Event",

        "Value":
            buyer_event_did[
                "Valid_Controls"
            ].median()
    },
])

summary.to_csv(
    OUTPUT_DIR
    / "07_V12A_1_EXECUTIVE_SUMMARY.csv",
    index=False
)

print("=" * 80)
print("V12A.1 COMPLETE")
print("=" * 80)

display(summary)



## Files to share back

Please send:

1. `07_V12A_1_EXECUTIVE_SUMMARY.csv`
2. `03_BUYER_DID_BY_SCENARIO.csv`
3. `04_PRE_BUYER_SUCCESS_VS_FAILURE.csv`
4. `05_BUYER_DID_MECHANISM_CORRELATIONS.csv`
5. `06_PRE_BUYER_FEATURE_RANKING.csv`
6. `02_BUYER_EVENT_LEVEL_DID.csv`

## Decision after V12A.1

If control-adjusted buyer changes show meaningful relationships with RPU / Conversion / Volume, the next step is V12B:

- add only PRE-event buyer features,
- keep post-event Buyer DiD only as mechanism evidence,
- retrain +$10 / +$15 / +$20 models,
- compare grouped held-out `Corporate_Change_ID` accuracy directly against V11.

If Buyer DiD remains weak, buyer lifecycle should stay a descriptive guardrail rather than being forced into the predictive model.
